# 02 — Application output regression

This notebook executes the application-level Yamada cases with the current canonical implementation and compares their canonical JSON representation with a frozen known-good SHA-256 digest.

It is self-contained: no historical branch is fetched, imported, or executed. The digest covers graph diagnostics, selected projections where relevant, PD codes, and exact Yamada polynomial strings.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'knotted_graph').exists():
    raise RuntimeError('Run this notebook from inside the KnottedGraph checkout.')
DRIVER = ROOT / 'dev' / 'application_yamada_regression.py'

import knotted_graph
from knotted_graph.invariants.yamada.native import native_available, native_import_error
from knotted_graph.invariants.yamada.factorized_frontier import native_factorized_available

print('Current notebook environment:', sys.executable)
print('Current KnottedGraph:', Path(knotted_graph.__file__).resolve())
print('Native resolved-graph Yamada backend:', native_available())
print('Factorized diagram Yamada backend:', native_factorized_available())
print('Native import error:', native_import_error())
assert native_available(), native_import_error()
assert native_factorized_available()

env = dict(os.environ)
env.pop('PYTHONPATH', None)
env['PYTHONNOUSERSITE'] = '1'
proc = subprocess.run([sys.executable, str(DRIVER)], cwd=ROOT, env=env, text=True, capture_output=True)
if proc.returncode:
    raise RuntimeError(f'Application regression driver failed.\nSTDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}')
current = json.loads(proc.stdout)
canonical = json.dumps(current, sort_keys=True, separators=(',', ':')).encode()
current_digest = hashlib.sha256(canonical).hexdigest()
EXPECTED_RECORDS = 38
EXPECTED_SHA256 = '07af584808160798429b589b782bd527caf72df3838cde0ae330903daa2a4898'
print('application records =', len(current))
print('canonical SHA-256 =', current_digest)


In [ ]:
assert len(current) == EXPECTED_RECORDS, (
    f'Application regression record count changed: {len(current)} != {EXPECTED_RECORDS}'
)
assert current_digest == EXPECTED_SHA256, (
    f'Application-level Yamada output changed: {current_digest} != {EXPECTED_SHA256}'
)
print('PASS: canonical application-level Yamada output matches the frozen golden digest.')


In [ ]:
physics = [row for row in current if row['application'] == 'physics']
mathematics = [row for row in current if row['application'] == 'mathematics']
print(f'Physics cases: {len(physics)}')
for row in physics:
    print(f"{row['case']:14s} gamma={row['gamma']:<4} V={row['nodes']:<3} E={row['edges']:<3} crossings={row['crossings']:<2} Yamada={row['yamada']}")
print(f'\nMathematics cases: {len(mathematics)}')
for row in mathematics:
    print(row['case'], row.get('yamada', row.get('yamada_negami')))


## Interpretation

A pass means the current canonical implementation reproduces the frozen 38-case application-level Yamada baseline exactly. The baseline is content-addressed by SHA-256 rather than tied to an obsolete Git branch.
